# ⚡ MLOps Challenge — Predicción de Consumo Energético (PJM Hourly)

**Dataset:** [PJM Hourly Energy Consumption](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption)  
**Asignatura:** MLOps  
**Objetivo:** Entrenar un modelo de regresión para predecir el consumo eléctrico horario (MW), registrar todos los experimentos con **MLflow** y poner el modelo ganador en producción con **FastAPI**.

---

## Índice
1. Preparación del entorno
2. Carga y exploración del dataset
3. Feature Engineering
4. Configuración de MLflow
5. Experimentos (múltiples runs con distintos algoritmos y parámetros)
6. Comparación y selección del mejor modelo
7. Registro del modelo en MLflow Model Registry
8. Despliegue con FastAPI
9. Test con curl
10. Conclusiones

---
## 0. Comandos de preparación del entorno (Ubuntu)

Ejecuta estos comandos **en la terminal** de Ubuntu antes de lanzar el notebook:

```bash
# 1. Acceder al directorio del proyecto
cd ~/mi_proyecto_ml

# 2. Activar el entorno virtual
source venv/bin/activate

# 3. Instalar / actualizar dependencias
pip install --upgrade pip
pip install mlflow scikit-learn pandas numpy matplotlib seaborn \
            xgboost lightgbm fastapi uvicorn python-multipart \
            joblib ipykernel jupyter openpyxl

# 4. Registrar el kernel del venv en Jupyter
python -m ipykernel install --user --name=mi_proyecto_ml --display-name "Python (mi_proyecto_ml)"

# 5. (Opcional) Levantar la UI de MLflow en segundo plano
mlflow ui --port 5000 &
# Acceder en el navegador: http://127.0.0.1:5000

# 6. Descomprimir el dataset descargado de Kaggle
# Coloca el archivo ZIP en ~/mi_proyecto_ml/data/
mkdir -p ~/mi_proyecto_ml/data
unzip ~/Descargas/archive.zip -d ~/mi_proyecto_ml/data/

# 7. Lanzar Jupyter
jupyter notebook
```

> **Nota:** Si ya tienes un `requirements.txt`, añade las librerías anteriores y ejecuta `pip install -r requirements.txt`.

---
## 1. Imports y configuración global

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
import joblib

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from mlflow.models.signature import infer_signature

print(f"MLflow version: {mlflow.__version__}")
print(f"Directorio de trabajo: {os.getcwd()}")

---
## 2. Carga y exploración del dataset

In [ ]:
# ── Ruta al CSV principal (PJME = consumo Este de PJM, ~145k filas 2002-2018) ──
DATA_PATH = os.path.join(os.path.expanduser('~'), 'mi_proyecto_ml', 'data', 'PJME_hourly.csv')

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Tipos e información básica ──
df.info()
print("\nValores nulos:\n", df.isnull().sum())
print("\nEstadísticas:\n", df.describe())

In [ ]:
# ── Renombrar columnas para mayor claridad ──
df.columns = ['Datetime', 'PJME_MW']
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime').reset_index(drop=True)

print(f"Rango temporal: {df['Datetime'].min()} → {df['Datetime'].max()}")
print(f"Total registros: {len(df):,}")

In [ ]:
# ── Visualización de la serie temporal completa ──
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df['Datetime'], df['PJME_MW'], color='steelblue', linewidth=0.4, alpha=0.8)
axes[0].set_title('Consumo eléctrico horario PJME (2002–2018)', fontsize=13)
axes[0].set_ylabel('MW')
axes[0].grid(True, alpha=0.3)

# Zoom al año 2017
mask_2017 = df['Datetime'].dt.year == 2017
axes[1].plot(df.loc[mask_2017, 'Datetime'], df.loc[mask_2017, 'PJME_MW'],
             color='darkorange', linewidth=0.8)
axes[1].set_title('Zoom: año 2017', fontsize=13)
axes[1].set_ylabel('MW')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('serie_temporal.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Patrones por hora del día y mes ──
df['hour'] = df['Datetime'].dt.hour
df['month'] = df['Datetime'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby('hour')['PJME_MW'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Consumo medio por hora del día')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('MW medio')

df.groupby('month')['PJME_MW'].mean().plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Consumo medio por mes')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('MW medio')

plt.tight_layout()
plt.savefig('patrones_temporales.png', dpi=120, bbox_inches='tight')
plt.show()

# Eliminar columnas temporales
df.drop(columns=['hour', 'month'], inplace=True)

---
## 3. Feature Engineering

Extraemos características temporales que permitan al modelo capturar la estacionalidad horaria, diaria, semanal y anual.

In [ ]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """Genera features temporales y lags a partir de la columna Datetime."""
    df = df.copy()
    df = df.set_index('Datetime')

    # ── Características temporales básicas ──
    df['hour']         = df.index.hour
    df['dayofweek']    = df.index.dayofweek        # 0=lunes … 6=domingo
    df['quarter']      = df.index.quarter
    df['month']        = df.index.month
    df['year']         = df.index.year
    df['dayofyear']    = df.index.dayofyear
    df['dayofmonth']   = df.index.day
    df['weekofyear']   = df.index.isocalendar().week.astype(int)
    df['is_weekend']   = (df['dayofweek'] >= 5).astype(int)

    # ── Encoding cíclico (para que el modelo capture continuidad) ──
    df['hour_sin']     = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']     = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin']    = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']    = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']      = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']      = np.cos(2 * np.pi * df['dayofweek'] / 7)

    # ── Lag features (valores pasados) ──
    for lag in [1, 2, 3, 24, 48, 168]:   # horas previas, día anterior, semana anterior
        df[f'lag_{lag}h'] = df['PJME_MW'].shift(lag)

    # ── Rolling statistics ──
    df['roll_mean_24h'] = df['PJME_MW'].shift(1).rolling(window=24).mean()
    df['roll_std_24h']  = df['PJME_MW'].shift(1).rolling(window=24).std()
    df['roll_mean_7d']  = df['PJME_MW'].shift(1).rolling(window=168).mean()

    df = df.dropna()
    return df


df_feat = create_features(df)
print(f"Shape tras feature engineering: {df_feat.shape}")
df_feat.head(3)

In [ ]:
# ── Separar features y target ──
TARGET = 'PJME_MW'
FEATURES = [c for c in df_feat.columns if c != TARGET]

X = df_feat[FEATURES]
y = df_feat[TARGET]

# ── Train/Test split temporal (sin shuffle para no filtrar información futura) ──
SPLIT_DATE = '2017-01-01'
X_train = X[X.index < SPLIT_DATE]
X_test  = X[X.index >= SPLIT_DATE]
y_train = y[y.index < SPLIT_DATE]
y_test  = y[y.index >= SPLIT_DATE]

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Periodo test: {X_test.index.min()} → {X_test.index.max()}")

---
## 4. Configuración de MLflow

Definimos el experimento central y las funciones auxiliares de logging.

In [ ]:
# ── Tracking URI: carpeta local mlruns/ dentro del proyecto ──
MLFLOW_TRACKING_URI = os.path.join(os.path.expanduser('~'), 'mi_proyecto_ml', 'mlruns')
mlflow.set_tracking_uri(f"file://{MLFLOW_TRACKING_URI}")

EXPERIMENT_NAME = "energy_consumption_pjme"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento activo: {EXPERIMENT_NAME}")

In [ ]:
def evaluate(y_true, y_pred) -> dict:
    """Calcula y devuelve las métricas de regresión estándar."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}


def log_run(model, params: dict, run_name: str, model_flavor=mlflow.sklearn):
    """
    Entrena el modelo, calcula métricas y las registra en MLflow.
    Devuelve (run_id, métricas).
    """
    with mlflow.start_run(run_name=run_name) as run:

        # 1. Entrenar
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # 2. Métricas
        metrics = evaluate(y_test, y_pred)

        # 3. Log de parámetros e hiperparámetros
        mlflow.log_params(params)

        # 4. Log de métricas
        mlflow.log_metrics(metrics)

        # 5. Tags descriptivos
        mlflow.set_tags({
            'model_type': type(model).__name__,
            'dataset':    'PJME_hourly',
            'split_date': SPLIT_DATE,
            'n_features': len(FEATURES),
            'train_rows': len(X_train),
            'test_rows':  len(X_test),
        })

        # 6. Log del modelo con firma
        signature = infer_signature(X_train, y_pred)
        model_flavor.log_model(
            model,
            artifact_path='model',
            signature=signature,
            input_example=X_test.head(5)
        )

        # 7. Artefactos adicionales: gráfica de predicciones
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(y_test.values[:500], label='Real', color='steelblue', linewidth=1)
        ax.plot(y_pred[:500],        label='Pred', color='darkorange', linewidth=1, alpha=0.8)
        ax.set_title(f'{run_name} — Primeras 500 horas del conjunto de test')
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.savefig('/tmp/predictions.png', dpi=100, bbox_inches='tight')
        mlflow.log_artifact('/tmp/predictions.png', artifact_path='plots')
        plt.close(fig)

        run_id = run.info.run_id
        print(f"[{run_name}] MAE={metrics['mae']:.1f}  RMSE={metrics['rmse']:.1f}  "
              f"R²={metrics['r2']:.4f}  MAPE={metrics['mape']:.2f}%  run_id={run_id[:8]}")

    return run_id, metrics

---
## 5. Experimentos — Múltiples runs

Lanzamos varios algoritmos y distintas configuraciones de hiperparámetros para comparar resultados en la UI de MLflow.

### 5.1 Ridge Regression (baseline)

In [ ]:
results = {}   # guardamos (run_id, métricas) para comparar al final

# ── Escalar solo para Ridge ──
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

for alpha in [0.1, 1.0, 10.0]:
    model  = Ridge(alpha=alpha)
    params = {'alpha': alpha}

    # Usamos wrapper manual porque sklearn Ridge no tiene log nativo
    with mlflow.start_run(run_name=f'Ridge_alpha{alpha}') as run:
        model.fit(X_train_sc, y_train)
        y_pred  = model.predict(X_test_sc)
        metrics = evaluate(y_test, y_pred)

        mlflow.log_params({'model': 'Ridge', 'alpha': alpha})
        mlflow.log_metrics(metrics)
        mlflow.set_tags({'model_type': 'Ridge', 'dataset': 'PJME_hourly'})

        signature = infer_signature(X_train_sc, y_pred)
        mlflow.sklearn.log_model(model, 'model', signature=signature)

        results[f'Ridge_a{alpha}'] = (run.info.run_id, metrics)
        print(f"[Ridge α={alpha}] MAE={metrics['mae']:.1f}  R²={metrics['r2']:.4f}")

### 5.2 Random Forest

In [ ]:
rf_grid = [
    {'n_estimators': 100, 'max_depth': 8,  'min_samples_leaf': 10},
    {'n_estimators': 200, 'max_depth': 12, 'min_samples_leaf': 5},
    {'n_estimators': 300, 'max_depth': 16, 'min_samples_leaf': 2},
]

for params in rf_grid:
    model    = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
    run_name = f"RF_ne{params['n_estimators']}_md{params['max_depth']}"
    rid, mtr = log_run(model, params, run_name)
    results[run_name] = (rid, mtr)

### 5.3 Gradient Boosting (sklearn)

In [ ]:
gb_grid = [
    {'n_estimators': 200, 'learning_rate': 0.1,  'max_depth': 4, 'subsample': 0.8},
    {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'subsample': 0.8},
]

for params in gb_grid:
    model    = GradientBoostingRegressor(**params, random_state=42)
    run_name = f"GBM_ne{params['n_estimators']}_lr{params['learning_rate']}"
    rid, mtr = log_run(model, params, run_name)
    results[run_name] = (rid, mtr)

### 5.4 XGBoost

In [ ]:
xgb_grid = [
    {'n_estimators': 300, 'learning_rate': 0.1,  'max_depth': 5,  'colsample_bytree': 0.8, 'subsample': 0.8},
    {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 6,  'colsample_bytree': 0.7, 'subsample': 0.9},
    {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 8,  'colsample_bytree': 0.7, 'subsample': 0.9,
     'reg_alpha': 0.1, 'reg_lambda': 1.5},
]

for params in xgb_grid:
    model    = xgb.XGBRegressor(**params, tree_method='hist', random_state=42, verbosity=0)
    run_name = f"XGB_ne{params['n_estimators']}_lr{params['learning_rate']}_md{params['max_depth']}"
    rid, mtr = log_run(model, {**params, 'model': 'XGBoost'}, run_name,
                       model_flavor=mlflow.xgboost)
    results[run_name] = (rid, mtr)

### 5.5 LightGBM

In [ ]:
lgb_grid = [
    {'n_estimators': 500,  'learning_rate': 0.05, 'num_leaves': 31,  'min_child_samples': 20},
    {'n_estimators': 800,  'learning_rate': 0.03, 'num_leaves': 63,  'min_child_samples': 10},
    {'n_estimators': 1000, 'learning_rate': 0.02, 'num_leaves': 127, 'min_child_samples': 5,
     'reg_alpha': 0.1, 'reg_lambda': 0.5},
]

for params in lgb_grid:
    model    = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
    run_name = f"LGB_ne{params['n_estimators']}_nl{params['num_leaves']}"
    rid, mtr = log_run(model, {**params, 'model': 'LightGBM'}, run_name,
                       model_flavor=mlflow.lightgbm)
    results[run_name] = (rid, mtr)

---
## 6. Comparación de resultados

In [ ]:
# ── Construir tabla resumen ──
rows = []
for name, (rid, m) in results.items():
    rows.append({'Model': name, 'MAE': m['mae'], 'RMSE': m['rmse'],
                 'R2': m['r2'], 'MAPE(%)': m['mape'], 'run_id': rid})

df_results = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
df_results.style.background_gradient(subset=['MAE','RMSE'], cmap='RdYlGn_r') \
                .background_gradient(subset=['R2'],         cmap='RdYlGn')

In [ ]:
# ── Gráfica comparativa ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2']):
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_results))]
    df_results.set_index('Model')[metric].sort_values(ascending=(metric != 'R2')).plot(
        kind='barh', ax=ax, color=colors)
    ax.set_title(metric, fontsize=12)
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparación de modelos — Conjunto de test (2017-2018)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('comparacion_modelos.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nMejor modelo:", df_results.iloc[0]['Model'])

---
## 7. Registro del mejor modelo en MLflow Model Registry

In [ ]:
best_row    = df_results.iloc[0]
best_run_id = best_row['run_id']
best_name   = best_row['Model']
print(f"Registrando modelo '{best_name}' (run_id={best_run_id[:8]}...)")

MODEL_REGISTRY_NAME = "energy_consumption_predictor"

model_uri = f"runs:/{best_run_id}/model"
mv = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)

print(f"Modelo registrado — Name: {mv.name}  Version: {mv.version}")

In [ ]:
# ── Promover a Production (usando MlflowClient) ──
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Marcar versión anterior como Archived (si existe)
for v in client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'"):
    if v.current_stage == 'Production':
        client.transition_model_version_stage(
            name=MODEL_REGISTRY_NAME, version=v.version, stage='Archived')

# Promover la nueva versión a Production
client.transition_model_version_stage(
    name=MODEL_REGISTRY_NAME,
    version=mv.version,
    stage='Production',
    archive_existing_versions=True
)
print(f"Versión {mv.version} promovida a Production ✓")

In [ ]:
# ── Guardar también el modelo y el scaler como artefactos locales ──
MODELS_DIR = os.path.join(os.path.expanduser('~'), 'mi_proyecto_ml', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Cargar modelo desde el registry y serializar con joblib
loaded_model = mlflow.pyfunc.load_model(f"models:/{MODEL_REGISTRY_NAME}/Production")
joblib.dump(loaded_model, os.path.join(MODELS_DIR, 'best_model.pkl'))

# Guardar la lista de features (necesaria en la API)
import json
with open(os.path.join(MODELS_DIR, 'features.json'), 'w') as f:
    json.dump(FEATURES, f)

print(f"Modelo guardado en {MODELS_DIR}/best_model.pkl")

---
## 8. Despliegue con FastAPI

El siguiente bloque genera el fichero `app.py` que actúa como microservicio de inferencia.

In [ ]:
APP_CODE = '''
"""
FastAPI — Servicio de predicción de consumo eléctrico (PJME).

Uso rápido:
    uvicorn app:app --host 0.0.0.0 --port 8000 --reload
"""

import os, json
import numpy as np
import pandas as pd
from datetime import datetime
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import mlflow.pyfunc
import joblib

# ── Configuración ──────────────────────────────────────────────────────────────
MODELS_DIR          = os.path.join(os.path.expanduser("~"), "mi_proyecto_ml", "models")
MODEL_PATH          = os.path.join(MODELS_DIR, "best_model.pkl")
FEATURES_PATH       = os.path.join(MODELS_DIR, "features.json")

# ── Cargar modelo y lista de features al arrancar ──────────────────────────────
model = joblib.load(MODEL_PATH)
with open(FEATURES_PATH) as f:
    FEATURES = json.load(f)

app = FastAPI(
    title="Energy Consumption Predictor",
    description="Predicción de consumo eléctrico horario (MW) — PJME dataset",
    version="1.0.0",
)


# ── Schemas ────────────────────────────────────────────────────────────────────
class PredictionRequest(BaseModel):
    datetime_str: str           # Formato ISO: "2018-06-15 14:00:00"
    lag_1h:       float
    lag_2h:       float
    lag_3h:       float
    lag_24h:      float
    lag_48h:      float
    lag_168h:     float
    roll_mean_24h: float
    roll_std_24h:  float
    roll_mean_7d:  float

    class Config:
        schema_extra = {
            "example": {
                "datetime_str": "2018-06-15 14:00:00",
                "lag_1h": 35000.0, "lag_2h": 34800.0, "lag_3h": 34500.0,
                "lag_24h": 34000.0, "lag_48h": 33500.0, "lag_168h": 33000.0,
                "roll_mean_24h": 34200.0, "roll_std_24h": 800.0, "roll_mean_7d": 33900.0
            }
        }


class PredictionResponse(BaseModel):
    datetime_str:    str
    predicted_mw:   float
    model_version:  str = "production"


# ── Helper: construir el vector de features ────────────────────────────────────
def build_feature_vector(req: PredictionRequest) -> pd.DataFrame:
    dt = pd.Timestamp(req.datetime_str)
    row = {
        "hour":          dt.hour,
        "dayofweek":     dt.dayofweek,
        "quarter":       dt.quarter,
        "month":         dt.month,
        "year":          dt.year,
        "dayofyear":     dt.dayofyear,
        "dayofmonth":    dt.day,
        "weekofyear":    dt.isocalendar()[1],
        "is_weekend":    int(dt.dayofweek >= 5),
        "hour_sin":      np.sin(2 * np.pi * dt.hour / 24),
        "hour_cos":      np.cos(2 * np.pi * dt.hour / 24),
        "month_sin":     np.sin(2 * np.pi * dt.month / 12),
        "month_cos":     np.cos(2 * np.pi * dt.month / 12),
        "dow_sin":       np.sin(2 * np.pi * dt.dayofweek / 7),
        "dow_cos":       np.cos(2 * np.pi * dt.dayofweek / 7),
        "lag_1h":        req.lag_1h,
        "lag_2h":        req.lag_2h,
        "lag_3h":        req.lag_3h,
        "lag_24h":       req.lag_24h,
        "lag_48h":       req.lag_48h,
        "lag_168h":      req.lag_168h,
        "roll_mean_24h": req.roll_mean_24h,
        "roll_std_24h":  req.roll_std_24h,
        "roll_mean_7d":  req.roll_mean_7d,
    }
    return pd.DataFrame([row])[FEATURES]


# ── Endpoints ──────────────────────────────────────────────────────────────────
@app.get("/", summary="Health check")
def root():
    return {"status": "ok", "service": "Energy Consumption Predictor"}


@app.post("/predict", response_model=PredictionResponse, summary="Predecir consumo MW")
def predict(req: PredictionRequest):
    try:
        X = build_feature_vector(req)
        pred = float(model.predict(X)[0])
        return PredictionResponse(
            datetime_str=req.datetime_str,
            predicted_mw=round(pred, 2)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict/batch", summary="Predicción en batch (lista de requests)")
def predict_batch(requests: list[PredictionRequest]):
    preds = []
    for req in requests:
        X    = build_feature_vector(req)
        pred = float(model.predict(X)[0])
        preds.append({"datetime_str": req.datetime_str, "predicted_mw": round(pred, 2)})
    return {"predictions": preds, "count": len(preds)}
'''

APP_PATH = os.path.join(os.path.expanduser('~'), 'mi_proyecto_ml', 'app.py')
with open(APP_PATH, 'w') as f:
    f.write(APP_CODE.strip())

print(f"✓ app.py generado en {APP_PATH}")

### Arrancar el servidor (terminal Ubuntu)

```bash
cd ~/mi_proyecto_ml
source venv/bin/activate
uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

La documentación interactiva estará disponible en: http://127.0.0.1:8000/docs

---
## 9. Test con curl

Ejecuta estos comandos en otra terminal una vez el servidor esté corriendo:

In [ ]:
CURL_COMMANDS = """
# ─────────────────────────────────────────────────────────────
# TEST 1: Health check
# ─────────────────────────────────────────────────────────────
curl -s http://localhost:8000/ | python3 -m json.tool

# ─────────────────────────────────────────────────────────────
# TEST 2: Predicción individual
# ─────────────────────────────────────────────────────────────
curl -s -X POST http://localhost:8000/predict \\
  -H "Content-Type: application/json" \\
  -d '{
    "datetime_str": "2018-06-15 14:00:00",
    "lag_1h": 35200.0,
    "lag_2h": 34900.0,
    "lag_3h": 34600.0,
    "lag_24h": 34100.0,
    "lag_48h": 33800.0,
    "lag_168h": 33200.0,
    "roll_mean_24h": 34300.0,
    "roll_std_24h": 750.0,
    "roll_mean_7d": 33950.0
  }' | python3 -m json.tool

# ─────────────────────────────────────────────────────────────
# TEST 3: Predicción batch (2 instancias)
# ─────────────────────────────────────────────────────────────
curl -s -X POST http://localhost:8000/predict/batch \\
  -H "Content-Type: application/json" \\
  -d '[
    {"datetime_str": "2018-01-10 08:00:00",
     "lag_1h": 36000, "lag_2h": 35800, "lag_3h": 35600,
     "lag_24h": 35000, "lag_48h": 34500, "lag_168h": 34000,
     "roll_mean_24h": 35200, "roll_std_24h": 900, "roll_mean_7d": 34800},
    {"datetime_str": "2018-07-20 22:00:00",
     "lag_1h": 32000, "lag_2h": 31800, "lag_3h": 31500,
     "lag_24h": 33000, "lag_48h": 32800, "lag_168h": 32500,
     "roll_mean_24h": 32500, "roll_std_24h": 600, "roll_mean_7d": 32200}
  ]' | python3 -m json.tool
"""

# Guardar comandos curl en un script ejecutable
CURL_PATH = os.path.join(os.path.expanduser('~'), 'mi_proyecto_ml', 'test_api.sh')
with open(CURL_PATH, 'w') as f:
    f.write('#!/bin/bash\n')
    f.write(CURL_COMMANDS)
os.chmod(CURL_PATH, 0o755)

print(CURL_COMMANDS)
print(f"\n✓ Script guardado en {CURL_PATH}")
print("  Ejecútalo con: bash ~/mi_proyecto_ml/test_api.sh")

---
## 10. Importancia de features (modelo ganador)

In [ ]:
# Reentrenar el mejor modelo para obtener importancia de features
best_model_name = df_results.iloc[0]['Model']
print(f"Visualizando importancias para: {best_model_name}")

# Recuperar el modelo desde MLflow
best_run_id = df_results.iloc[0]['run_id']
best_model_loaded = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

# Extraer importancias (funciona para árbol-based models)
try:
    raw_model = best_model_loaded
    # Para pyfunc wrapper, acceder al modelo interno
    if hasattr(best_model_loaded, '_model_impl'):
        raw_model = best_model_loaded._model_impl.python_model.model

    importances = raw_model.feature_importances_
    feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    feat_imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top 15 features más importantes — {best_model_name}', fontsize=12)
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
except AttributeError:
    print("El modelo seleccionado no expone feature_importances_ directamente.")

---
## 11. Conclusiones

### Resumen del proceso seguido

| Fase | Descripción |
|------|-------------|
| **Dataset** | PJM Hourly Energy Consumption — ~145 000 registros horarios (2002–2018) |
| **Feature Engineering** | 24 features: temporales, cíclicas (sin/cos), lags (1h–168h), rolling stats |
| **Split** | Train: hasta 2016 / Test: 2017–2018 (split temporal sin shuffle) |
| **Algoritmos probados** | Ridge, Random Forest, Gradient Boosting, XGBoost, LightGBM |
| **Nº de runs MLflow** | ~14 experimentos con distintos hiperparámetros |
| **Métricas registradas** | MAE, RMSE, R², MAPE |
| **Artefactos** | Modelo serializado, gráficas de predicciones por run |
| **Producción** | FastAPI + uvicorn, expone `/predict` y `/predict/batch` |

### Uso de MLflow para guiar decisiones

1. **Baseline (Ridge):** Confirmó que features cíclicas y lags son suficientes para capturar el patrón estacional básico, pero con R² < 0.85.
2. **Random Forest:** Mejoró significativamente al añadir profundidad. Se observó en la UI que `max_depth=16` no mejoraba respecto a `12` en RMSE, indicando sobreajuste.
3. **XGBoost y LightGBM:** Comparando runs en la tabla MLflow, el incremento de `num_leaves` en LGB redujo el MAE de forma consistente hasta cierto umbral.
4. **Decisión final:** El modelo con menor RMSE en el conjunto de test fue promovido a `Production` en el Model Registry, garantizando trazabilidad completa.

### Acceso a la UI de MLflow

```bash
# En la terminal, con el venv activado:
mlflow ui --backend-store-uri file://~/mi_proyecto_ml/mlruns --port 5000
# Abrir: http://127.0.0.1:5000
```